In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- 1. HYPERPARAMETERS & DATA ---
torch.manual_seed(42)
block_size = 16      # Context length (how many characters to look back)
batch_size = 8       # Number of independent sequences processed in parallel
n_embd = 32          # Embedding dimension
n_head = 4           # Number of attention heads
n_layer = 3          # Number of Transformer blocks stacked together
learning_rate = 1e-3
epochs = 3500

# Training dataset
text = "this is a tiny dataset for our mini language model. building an llm from scratch is highly rewarding! it learns patterns character by character. "
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Character-level tokenizer mappings
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
data = torch.tensor(encode(text), dtype=torch.long)

def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# --- 2. ARCHITECTURE COMPONENTS ---
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, n_embd):
        super().__init__()
        self.n_head = num_heads
        self.n_embd = n_embd
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# --- 3. THE COMPLETE GPT MODEL ---
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        
        x = tok_emb + pos_emb 
        x = self.blocks(x) 
        x = self.ln_f(x) 
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
                
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# --- 4. MODEL INITIALIZATION & TRAINING ---
model = GPTLanguageModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("--- UNTRAINED MODEL GENERATION ---")
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))

print("\nStarting training...")
for iter in range(epochs):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if (iter + 1) % 500 == 0:
        print(f"Step {iter + 1:4d} | Loss: {loss.item():.4f}")

print("\n--- FULLY TRAINED MODEL GENERATION (WITH TEMPERATURE & TOP-K) ---")
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(context, max_new_tokens=200, temperature=1.1, top_k=3)[0].tolist()))

--- UNTRAINED MODEL GENERATION ---
 ntlbifebluyhlalmwe.ewmtmgellnlnrldrhsesse.bg.nmy. nts ysotw.sdb yy by.rew mduylmwrburcs.calulwebutin

Starting training...
Step  500 | Loss: 0.2369
Step 1000 | Loss: 0.1184
Step 1500 | Loss: 0.1549
Step 2000 | Loss: 0.1716
Step 2500 | Loss: 0.1544
Step 3000 | Loss: 0.1321
Step 3500 | Loss: 0.1399

--- FULLY TRAINED MODEL GENERATION (WITH TEMPERATURE & TOP-K) ---
 is a tiny dataset for our mini language model. building an llm from scratch is highly rewarding! it learns patterns character by character. by character. buins pat terns character by character. by cha
